In [ ]:
### Cell 1 — locate the dataset and load its tables

from pathlib import Path
import random

import numpy as np
import pandas as pd
import torch

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Kaggle may expose competition data in either location.
data_candidates = [
    Path("/kaggle/input/rsna-knee-abnormality-detection"),
    Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
]

DATA = next((path for path in data_candidates if path.is_dir()), None)

if DATA is None:
    raise FileNotFoundError(
        "Competition data was not found. Attach "
        "'rsna-knee-abnormality-detection' to this notebook."
    )

csv_paths = sorted(DATA.glob("*.csv"))

print("Dataset directory:", DATA)
print("CSV files:", [path.name for path in csv_paths])
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

train = pd.read_csv(DATA / "train.csv")
train_series = pd.read_csv(DATA / "train_series.csv")
test = pd.read_csv(DATA / "test.csv")
test_series = pd.read_csv(DATA / "test_series.csv")
sample_submission = pd.read_csv(DATA / "sample_submission.csv")

print()
print("train:", train.shape)
print("train_series:", train_series.shape)
print("test:", test.shape)
print("test_series:", test_series.shape)
print("sample_submission:", sample_submission.shape)

display(train.head(3))
display(train_series.head(3))

In [ ]:
### Select a report-only study with the widest plane coverage

TARGETS = [column for column in sample_submission.columns if column != "StudyInstanceUID"]
train["StudyInstanceUID"] = train["StudyInstanceUID"].astype(str)
train_series["StudyInstanceUID"] = train_series["StudyInstanceUID"].astype(str)
train_series["SeriesInstanceUID"] = train_series["SeriesInstanceUID"].astype(str)

report_only_ids = set(
    train.loc[train[TARGETS].isna().all(axis=1), "StudyInstanceUID"]
)
planes = {"Axial", "Coronal", "Sagittal"}
best_rows = None
best_study = None
best_score = -1

for study_uid in sorted(report_only_ids):
    group = train_series[
        (train_series["StudyInstanceUID"] == study_uid)
        & train_series["Anatomical_Plane"].isin(planes)
    ].copy()
    if group.empty:
        continue
    group = group.sort_values(
        ["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression", "SeriesInstanceUID"],
        ascending=[True, False, False, True],
    )
    selected = group.groupby("Anatomical_Plane", as_index=False).head(1)
    score = selected["Anatomical_Plane"].nunique()
    if score > best_score:
        best_rows, best_study, best_score = selected, study_uid, score
    if score == 3:
        break

if best_rows is None or best_score < 2:
    raise RuntimeError("Could not find a report-only study with at least two standard planes")

example_study_uid = best_study
example_series = best_rows.sort_values("Anatomical_Plane").reset_index(drop=True)
representative_rows = example_series
print("Report-only study selected; no official labels are read.")
print("Series selected by plane:")
display(example_series[["Anatomical_Plane", "Fluid_Sensitive", "Fat_Suppression"]])


In [ ]:
%pip install -q pydicom==3.0.1

import pydicom
import matplotlib.pyplot as plt

TRAIN_DICOM_ROOT = DATA / "train_series"
assert TRAIN_DICOM_ROOT.is_dir(), TRAIN_DICOM_ROOT


In [ ]:
### Cell 5 — reconstruct and visualize one real volume

import matplotlib.pyplot as plt


def reconstruct_series(study_uid, series_uid):
  """
  Reconstruct one classic single-frame MRI series.

  Returns
  -------
  volume:
      Float32 NumPy array with axes (slice, row, column).
  geometry:
      Spatial metadata describing the reconstructed array.
  """
  folder = (
      TRAIN_DICOM_ROOT
      / str(study_uid)
      / str(series_uid)
  )

  dicom_paths = sorted(folder.glob("*.dcm"))

  if len(dicom_paths) < 2:
      raise ValueError(
          f"Expected at least two DICOM slices: {folder}"
      )

  headers = [
      pydicom.dcmread(path, stop_before_pixels=True)
      for path in dicom_paths
  ]

  reference_orientation = np.asarray(
      headers[0].ImageOrientationPatient,
      dtype=np.float64,
  )

  orientations = np.asarray(
      [
          header.ImageOrientationPatient
          for header in headers
      ],
      dtype=np.float64,
  )

  if not np.allclose(
      orientations,
      reference_orientation,
      atol=1e-4,
      rtol=0,
  ):
      raise ValueError("Orientation varies within the series")

  pixel_spacing = np.asarray(
      headers[0].PixelSpacing,
      dtype=np.float64,
  )

  for header in headers:
      current_spacing = np.asarray(
          header.PixelSpacing,
          dtype=np.float64,
      )

      if not np.allclose(
          current_spacing,
          pixel_spacing,
          atol=1e-4,
          rtol=0,
      ):
          raise ValueError(
              "PixelSpacing varies within the series"
          )

  # DICOM IOP stores the directions across columns and rows.
  column_axis_direction = reference_orientation[:3]
  row_axis_direction = reference_orientation[3:]

  slice_normal = np.cross(
      column_axis_direction,
      row_axis_direction,
  )

  positions = np.asarray(
      [
          header.ImagePositionPatient
          for header in headers
      ],
      dtype=np.float64,
  )

  projections = positions @ slice_normal
  physical_order = np.argsort(
      projections,
      kind="stable",
  )

  ordered_positions = positions[physical_order]
  ordered_projections = projections[physical_order]
  slice_gaps = np.diff(ordered_projections)

  if np.any(slice_gaps <= 1e-3):
      raise ValueError(
          "Repeated or indistinguishable slice positions"
      )

  slice_spacing = float(np.median(slice_gaps))

  if not np.allclose(
      slice_gaps,
      slice_spacing,
      atol=0.05,
      rtol=0.01,
  ):
      raise ValueError(
          "Irregular spacing or missing slices"
      )

  ordered_slices = []
  ordered_files = []

  for original_index in physical_order:
      path = dicom_paths[original_index]
      dcm = pydicom.dcmread(path)

      pixels = dcm.pixel_array.astype(np.float32)

      if pixels.shape != (
          int(dcm.Rows),
          int(dcm.Columns),
      ):
          raise ValueError(
              f"Unexpected pixel shape: {pixels.shape}"
          )

      # Exclude declared padding from intensity statistics.
      if hasattr(dcm, "PixelPaddingValue"):
          padding_value = float(dcm.PixelPaddingValue)

          if hasattr(dcm, "PixelPaddingRangeLimit"):
              padding_limit = float(
                  dcm.PixelPaddingRangeLimit
              )
          else:
              padding_limit = padding_value

          padding_low = min(
              padding_value,
              padding_limit,
          )
          padding_high = max(
              padding_value,
              padding_limit,
          )

          padding_mask = (
              (pixels >= padding_low)
              & (pixels <= padding_high)
          )
      else:
          padding_mask = np.zeros(
              pixels.shape,
              dtype=bool,
          )

      slope = float(
          getattr(dcm, "RescaleSlope", 1.0)
      )
      intercept = float(
          getattr(dcm, "RescaleIntercept", 0.0)
      )

      pixels = pixels * slope + intercept
      pixels[padding_mask] = np.nan

      if not np.isfinite(pixels).any():
          raise ValueError(
              f"No finite pixels in {path.name}"
          )

      ordered_slices.append(pixels)
      ordered_files.append(str(path))

  volume = np.stack(
      ordered_slices,
      axis=0,
  ).astype(np.float32)

  # Array axes are (slice, row, column).
  # This affine maps those indices into DICOM LPS millimetres.
  affine = np.eye(4, dtype=np.float64)
  affine[:3, 0] = (
      slice_normal * slice_spacing
  )
  affine[:3, 1] = (
      row_axis_direction * pixel_spacing[0]
  )
  affine[:3, 2] = (
      column_axis_direction * pixel_spacing[1]
  )
  affine[:3, 3] = ordered_positions[0]

  geometry = {
      "array_axes": ("slice", "row", "column"),
      "coordinate_system": "DICOM LPS millimetres",
      "spacing_drc_mm": (
          slice_spacing,
          float(pixel_spacing[0]),
          float(pixel_spacing[1]),
      ),
      "affine_array_to_lps": affine,
      "slice_normal_lps": slice_normal,
      "ordered_positions_lps": ordered_positions,
      "ordered_files": ordered_files,
      "photometric_interpretation": str(
          getattr(
              headers[0],
              "PhotometricInterpretation",
              "",
          )
      ),
  }

  return volume, geometry




In [ ]:
### Cell 6 — compare sagittal, coronal, and axial reconstructions

# Prefer a fluid-sensitive series for each anatomical plane.
representative_rows = (
  example_series
  .sort_values(
      ["Anatomical_Plane", "Fluid_Sensitive"],
      ascending=[True, False],
  )
  .groupby("Anatomical_Plane", as_index=False)
  .head(1)
  .sort_values("Anatomical_Plane")
  .reset_index(drop=True)
)

fig, axes = plt.subplots(
  len(representative_rows),
  3,
  figsize=(13, 4 * len(representative_rows)),
)

reconstruction_summary = []

for row_index, series_row in representative_rows.iterrows():
  plane = series_row["Anatomical_Plane"]

  plane_volume, plane_geometry = reconstruct_series(
      series_row["StudyInstanceUID"],
      series_row["SeriesInstanceUID"],
  )

  (
      plane_slice_spacing,
      plane_row_spacing,
      plane_column_spacing,
  ) = plane_geometry["spacing_drc_mm"]

  finite = plane_volume[
      np.isfinite(plane_volume)
  ]

  window_low, window_high = np.percentile(
      finite,
      [1, 99],
  )

  middle_slice = plane_volume.shape[0] // 2
  middle_row = plane_volume.shape[1] // 2
  middle_column = plane_volume.shape[2] // 2

  # Native acquisition plane.
  axes[row_index, 0].imshow(
      plane_volume[middle_slice],
      cmap="gray",
      vmin=window_low,
      vmax=window_high,
      aspect=(
          plane_row_spacing
          / plane_column_spacing
      ),
  )
  axes[row_index, 0].set_title(
      f"{plane}: native centre slice"
  )
  axes[row_index, 0].set_xlabel("column")
  axes[row_index, 0].set_ylabel("row")

  # Orthogonal centre-row view.
  axes[row_index, 1].imshow(
      plane_volume[:, middle_row, :],
      cmap="gray",
      vmin=window_low,
      vmax=window_high,
      aspect=(
          plane_slice_spacing
          / plane_column_spacing
      ),
  )
  axes[row_index, 1].set_title(
      f"{plane}: centre row"
  )
  axes[row_index, 1].set_xlabel("column")
  axes[row_index, 1].set_ylabel(
      "physically ordered slice"
  )

  # Orthogonal centre-column view.
  axes[row_index, 2].imshow(
      plane_volume[:, :, middle_column],
      cmap="gray",
      vmin=window_low,
      vmax=window_high,
      aspect=(
          plane_slice_spacing
          / plane_row_spacing
      ),
  )
  axes[row_index, 2].set_title(
      f"{plane}: centre column"
  )
  axes[row_index, 2].set_xlabel("row")
  axes[row_index, 2].set_ylabel(
      "physically ordered slice"
  )

  reconstruction_summary.append(
      {
          "plane": plane,
          "fluid_sensitive":
              series_row["Fluid_Sensitive"],
          "fat_suppression":
              series_row["Fat_Suppression"],
          "shape_drc": tuple(
              plane_volume.shape
          ),
          "spacing_drc_mm":
              plane_geometry["spacing_drc_mm"],
          "physical_fov_mm": tuple(
              round(size * spacing, 2)
              for size, spacing in zip(
                  plane_volume.shape,
                  plane_geometry[
                      "spacing_drc_mm"
                  ],
              )
          ),
      }
  )

plt.tight_layout()
plt.show()

display(
  pd.DataFrame(reconstruction_summary)
)

## Visual review gate

Inspect the reconstruction figure in the preceding cell before continuing. If any view is blank, truncated, or incorrectly oriented, stop and fix preprocessing first. Run the remaining cells individually after confirming the images are usable. The selected study is report-only; the notebook never reads official labels.


## Load the frozen Pillar-0 checkpoint

This cell uses the gated checkpoint at the revision already verified by the project. It performs inference only.

In [ ]:
### Cell 8 — verify Pillar-0 checkpoint access

%pip install -q \
  "transformers==4.56.2" \
  "huggingface_hub>=0.34,<1.0" \
  "accelerate==1.10.1"


In [ ]:
import transformers
from huggingface_hub import HfApi
from kaggle_secrets import UserSecretsClient

MODEL_ID = "YalaLab/Pillar0-BreastMRI"
MODEL_REVISION = "main"

try:
  hf_token = (
      UserSecretsClient()
      .get_secret("HF_TOKEN")
  )
except Exception as error:
  raise RuntimeError(
      "Kaggle secret HF_TOKEN is missing or "
      "not enabled for this notebook."
  ) from error

if not hf_token:
  raise RuntimeError(
      "Kaggle secret HF_TOKEN is empty."
  )

hf_api = HfApi(token=hf_token)

try:
  model_info = hf_api.model_info(
      MODEL_ID,
      revision=MODEL_REVISION,
      files_metadata=True,
  )
except Exception as error:
  raise RuntimeError(
      "The Hugging Face token could not access "
      f"{MODEL_ID}. Confirm that the account has "
      "accepted the gated model conditions."
  ) from error

model_files = pd.DataFrame(
  [
      {
          "filename": file.rfilename,
          "size_mb": (
              file.size / 1024**2
              if file.size is not None
              else np.nan
          ),
      }
      for file in model_info.siblings
  ]
).sort_values("filename")

print("Transformers version:", transformers.__version__)
print("Model:", model_info.id)
print("Resolved revision:", model_info.sha)
print("Gated:", model_info.gated)
print(
  "Total repository size (MB):",
  round(
      model_files["size_mb"].sum(
          skipna=True
      ),
      1,
  ),
)

display(model_files)


In [ ]:
import json

from transformers import AutoModel

PILLAR_REVISION = (
  "be5050ba482607f90e607e044f03a2ebc38f98c3"
)

if not torch.cuda.is_available():
  raise RuntimeError(
      "Enable a Kaggle GPU before loading Pillar-0."
  )

device = torch.device("cuda")

# Make the cell safe to rerun without loading a second model.
if "pillar" not in globals():
  torch.cuda.empty_cache()

  pillar = AutoModel.from_pretrained(
      MODEL_ID,
      revision=PILLAR_REVISION,
      token=hf_token,
      trust_remote_code=True,
      low_cpu_mem_usage=True,
  )

pillar = pillar.eval().to(device)

for parameter in pillar.parameters():
  parameter.requires_grad_(False)

parameter_count = sum(
  parameter.numel()
  for parameter in pillar.parameters()
)

trainable_parameter_count = sum(
  parameter.numel()
  for parameter in pillar.parameters()
  if parameter.requires_grad
)

first_parameter = next(
  pillar.parameters()
)

print("Model class:", type(pillar))
print("Parameters:", f"{parameter_count:,}")
print(
  "Trainable parameters:",
  trainable_parameter_count,
)
print("Training mode:", pillar.training)
print("Device:", first_parameter.device)
print("Parameter dtype:", first_parameter.dtype)
print(
  "GPU memory allocated (MB):",
  round(
      torch.cuda.memory_allocated()
      / 1024**2,
      1,
  ),
)
print(
  "GPU memory reserved (MB):",
  round(
      torch.cuda.memory_reserved()
      / 1024**2,
      1,
  ),
)

print()
print("Model configuration:")
print(
  json.dumps(
      pillar.config.to_dict(),
      indent=2,
      default=str,
  )
)



In [ ]:
### Cell 10 — canonicalize and preprocess one series

from itertools import product

from scipy.ndimage import affine_transform


TARGET_SPACING_XYZ_MM = np.array(
  [1.0, 1.0, 1.0],
  dtype=np.float64,
)

TARGET_SHAPE_ZYX = (
  192,
  384,
  384,
)


def canonical_resample_lps(
  native_volume,
  native_affine,
  target_spacing_xyz_mm=TARGET_SPACING_XYZ_MM,
):
  """Resample signal and validity together, avoiding padding bleed.

  Returns intensity normalized by interpolated support, a fractional support
  map in [0,1], and canonical-grid metadata. A genuine zero remains supported.
  """
  native_shape = np.asarray(native_volume.shape, dtype=int)
  native_validity = np.isfinite(native_volume).astype(np.float32)
  native_signal = np.where(
      native_validity > 0,
      np.nan_to_num(native_volume, nan=0.0, posinf=0.0, neginf=0.0),
      0.0,
  ).astype(np.float32)

  native_corners = np.asarray(list(product(
      [0, native_shape[0] - 1],
      [0, native_shape[1] - 1],
      [0, native_shape[2] - 1],
  )), dtype=np.float64)
  homogeneous_corners = np.concatenate(
      [native_corners, np.ones((len(native_corners), 1), dtype=np.float64)], axis=1
  )
  lps_corners = (native_affine @ homogeneous_corners.T).T[:, :3]
  lps_min = lps_corners.min(axis=0)
  lps_max = lps_corners.max(axis=0)
  physical_extent_xyz = lps_max - lps_min
  output_size_xyz = np.ceil(physical_extent_xyz / target_spacing_xyz_mm).astype(int) + 1
  output_shape_zyx = tuple(output_size_xyz[::-1])

  canonical_affine = np.eye(4, dtype=np.float64)
  canonical_affine[:3, 0] = [0, 0, target_spacing_xyz_mm[2]]
  canonical_affine[:3, 1] = [0, target_spacing_xyz_mm[1], 0]
  canonical_affine[:3, 2] = [target_spacing_xyz_mm[0], 0, 0]
  canonical_affine[:3, 3] = lps_min
  output_to_input = np.linalg.inv(native_affine) @ canonical_affine
  transform = dict(
      matrix=output_to_input[:3, :3],
      offset=output_to_input[:3, 3],
      output_shape=output_shape_zyx,
      order=1,
      mode="constant",
      cval=0.0,
      prefilter=False,
  )

  numerator = affine_transform(native_signal, **transform).astype(np.float32)
  support = affine_transform(native_validity, **transform).astype(np.float32)
  support = np.clip(support, 0.0, 1.0)
  canonical_volume = np.full(output_shape_zyx, np.nan, dtype=np.float32)
  supported = support > 1e-6
  canonical_volume[supported] = numerator[supported] / support[supported]

  metadata = {
      "input_shape_src": tuple(native_volume.shape),
      "output_shape_zyx": tuple(canonical_volume.shape),
      "output_spacing_xyz_mm": tuple(target_spacing_xyz_mm.tolist()),
      "output_origin_lps": tuple(lps_min.tolist()),
      "physical_extent_xyz_mm": tuple(physical_extent_xyz.tolist()),
      "canonical_affine_zyx_to_lps": canonical_affine,
  }
  return canonical_volume, support, metadata


def high_contrast_mr_window(
  volume,
  support,
  lower_percentile=20,
  upper_percentile=99,
  support_threshold=0.5,
):
  """Window supported voxels only; set unsupported output voxels to zero."""
  tensor = torch.as_tensor(volume, dtype=torch.float32)
  support_tensor = torch.as_tensor(support, dtype=torch.float32)
  if tensor.shape != support_tensor.shape:
      raise ValueError(f"Volume/support shape mismatch: {tensor.shape} vs {support_tensor.shape}")
  core = torch.isfinite(tensor) & (support_tensor >= support_threshold)
  finite_values = tensor[core]
  if finite_values.numel() == 0:
      raise ValueError("No supported voxels available for MRI intensity window")
  lower = torch.quantile(finite_values, lower_percentile / 100)
  upper = torch.quantile(finite_values, upper_percentile / 100)
  if upper <= lower:
      raise ValueError("Degenerate MRI intensity window")
  safe_tensor = torch.where(torch.isfinite(tensor), tensor, lower)
  windowed = ((safe_tensor - lower) / (upper - lower)).clamp(0, 1)
  has_support = torch.isfinite(tensor) & (support_tensor > 1e-6)
  windowed = torch.where(has_support, windowed, torch.zeros_like(windowed))
  return windowed, float(lower), float(upper)


def center_pad_or_crop_zyx(
  volume,
  target_shape_zyx,
):
  """
  Centre pad or crop a (z,y,x) tensor.

  Returns the processed volume and a record of each transformation.
  """
  output = torch.zeros(
      target_shape_zyx,
      dtype=volume.dtype,
  )

  source_slices = []
  destination_slices = []
  transformations = []

  for axis_name, current, target in zip(
      ("z", "y", "x"),
      volume.shape,
      target_shape_zyx,
  ):
      if current <= target:
          source_start = 0
          source_end = current

          destination_start = (
              target - current
          ) // 2
          destination_end = (
              destination_start + current
          )

          transformations.append(
              {
                  "axis": axis_name,
                  "operation": "pad",
                  "before": destination_start,
                  "after": (
                      target
                      - destination_end
                  ),
              }
          )
      else:
          source_start = (
              current - target
          ) // 2
          source_end = (
              source_start + target
          )

          destination_start = 0
          destination_end = target

          transformations.append(
              {
                  "axis": axis_name,
                  "operation": "crop",
                  "start": source_start,
                  "end": source_end,
              }
          )

      source_slices.append(
          slice(
              source_start,
              source_end,
          )
      )
      destination_slices.append(
          slice(
              destination_start,
              destination_end,
          )
      )

  output[
      tuple(destination_slices)
  ] = volume[
      tuple(source_slices)
  ]

  return output, transformations




In [ ]:
### Frozen real-volume representation audit (no training)

import gc
import json
import numpy as np
import torch.nn.functional as F

if pillar.training or any(p.requires_grad for p in pillar.parameters()):
    raise RuntimeError("Expected frozen Pillar-0 in eval mode")

MODEL_INPUT_KEY = "breast_mr"
OUTPUT_DIR = Path("/kaggle/working/pillar0_spatial_representation_audit")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


def shape_tree(value):
    if isinstance(value, torch.Tensor):
        return [list(value.shape)]
    if isinstance(value, (tuple, list)):
        return [shape for item in value for shape in shape_tree(item)]
    if isinstance(value, dict):
        return [shape for item in value.values() for shape in shape_tree(item)]
    return []


def summarize_energy(energy, support, threshold_kind):
    flat_e = energy.reshape(-1)
    flat_s = support.reshape(-1)
    if threshold_kind == "supported_core":
        mask = flat_s >= 0.95
    elif threshold_kind == "mostly_padding":
        mask = flat_s <= 0.05
    else:
        mask = (flat_s > 0.05) & (flat_s < 0.95)
    values = flat_e[mask]
    if values.numel() == 0:
        return {"cells": 0, "mean_rms": None, "median_rms": None, "p95_rms": None}
    return {
        "cells": int(values.numel()),
        "mean_rms": float(values.mean().cpu()),
        "median_rms": float(values.median().cpu()),
        "p95_rms": float(torch.quantile(values.float(), 0.95).cpu()),
    }


visual = pillar.model.visual
patch_module = visual.patch_embeds[MODEL_INPUT_KEY]
pooled_module = visual.maxpool
atlas_stages = list(visual.atlas_models)

series_summaries = []
final_embeddings = []
scale_vectors_by_series = []
plot_payload = []

for row in representative_rows.itertuples(index=False):
    plane = str(row.Anatomical_Plane)
    print(f"Reconstructing and encoding {plane} series", flush=True)
    native_volume, native_geometry = reconstruct_series(
        row.StudyInstanceUID, row.SeriesInstanceUID
    )
    canonical, support_canonical, canonical_info = canonical_resample_lps(
        native_volume, native_geometry["affine_array_to_lps"]
    )
    windowed, window_low, window_high = high_contrast_mr_window(
        canonical, support_canonical
    )
    padded, spatial_operations = center_pad_or_crop_zyx(windowed, TARGET_SHAPE_ZYX)
    support_padded, _ = center_pad_or_crop_zyx(
        torch.from_numpy(support_canonical), TARGET_SHAPE_ZYX
    )

    # Model input order matches the verified Kaggle pipeline: (H,W,D)=(y,x,z),
    # with the single acquisition repeated into the checkpoint's three channels.
    model_hwd = padded.permute(1, 2, 0).contiguous()
    support_hwd = support_padded.permute(1, 2, 0).contiguous().clamp_(0, 1)
    model_tensor = model_hwd[None, None].expand(1, 3, *model_hwd.shape)
    gpu_tensor = model_tensor.contiguous().to(device=device, dtype=torch.float32)
    support_gpu = support_hwd[None, None].to(device=device, dtype=torch.float32)

    captured = {"stage_shapes": {}, "pooled_scales": [], "patch": None}

    def patch_hook(_module, _inputs, output):
        if not isinstance(output, torch.Tensor) or output.ndim != 5:
            raise RuntimeError(f"Unexpected patch-grid output: {type(output)}")
        support_grid = F.adaptive_avg_pool3d(support_gpu, output.shape[-3:])[0, 0]
        energy = output.detach().float().square().mean(dim=1).sqrt()[0]
        captured["patch"] = {
            "shape": list(output.shape),
            "support_shape": list(support_grid.shape),
            "valid_core": summarize_energy(energy, support_grid, "supported_core"),
            "boundary": summarize_energy(energy, support_grid, "boundary"),
            "mostly_padding": summarize_energy(energy, support_grid, "mostly_padding"),
            "energy_cpu": energy.cpu(),
            "support_cpu": support_grid.cpu(),
        }

    def pooled_hook(_module, _inputs, output):
        if isinstance(output, torch.Tensor):
            captured["pooled_scales"].append(output.detach().float().cpu().reshape(-1))

    handles = [patch_module.register_forward_hook(patch_hook), pooled_module.register_forward_hook(pooled_hook)]
    for stage_index, stage in enumerate(atlas_stages):
        def stage_hook(_module, _inputs, output, index=stage_index):
            captured["stage_shapes"][f"atlas_stage_{index}"] = shape_tree(output)
        handles.append(stage.register_forward_hook(stage_hook))

    torch.cuda.synchronize()
    with torch.inference_mode():
        output = pillar.extract_vision_feats({MODEL_INPUT_KEY: gpu_tensor})
    torch.cuda.synchronize()
    for handle in handles:
        handle.remove()

    if captured["patch"] is None or len(captured["pooled_scales"]) != 3:
        raise RuntimeError(
            f"Expected patch grid and three scale readouts; got "
            f"patch={captured['patch'] is not None}, pools={len(captured['pooled_scales'])}"
        )
    embedding = output.detach().float().cpu().reshape(-1)
    if embedding.shape != (1152,) or not torch.isfinite(embedding).all():
        raise RuntimeError(f"Invalid final embedding: {tuple(embedding.shape)}")

    scale_vectors = torch.stack(captured["pooled_scales"])
    reconstructed_final = F.normalize(scale_vectors.reshape(1, -1), dim=-1).reshape(-1)
    pool_match_error = float((reconstructed_final - embedding).abs().max())
    scale_norms = torch.linalg.vector_norm(scale_vectors, dim=-1)
    scale_shares = (scale_norms / scale_norms.sum().clamp_min(1e-12)).tolist()
    patch = captured["patch"]

    series_summaries.append({
        "plane": plane,
        "native_shape_drc": list(native_volume.shape),
        "native_finite_voxel_fraction": float(np.isfinite(native_volume).mean()),
        "native_nan_or_infinite_voxel_fraction": float(1.0 - np.isfinite(native_volume).mean()),
        "canonical_fraction_with_support": float(np.mean(support_canonical > 1e-6)),
        "native_spacing_drc_mm": [float(v) for v in native_geometry["spacing_drc_mm"]],
        "canonical_shape_zyx": list(canonical.shape),
        "model_input_shape_bchwd": list(model_tensor.shape),
        "valid_dicom_support_fraction": float(support_hwd.mean()),
        "window_percentiles_20_99": [float(window_low), float(window_high)],
        "spatial_operations": spatial_operations,
        "patch_grid_shape": patch["shape"],
        "patch_grid_energy_by_support": {
            key: patch[key] for key in ["valid_core", "boundary", "mostly_padding"]
        },
        "atlas_stage_output_shapes": captured["stage_shapes"],
        "pooled_scale_vector_shapes": [list(v.shape) for v in captured["pooled_scales"]],
        "pooled_scale_norm_shares": [float(v) for v in scale_shares],
        "final_embedding_shape": list(embedding.shape),
        "final_embedding_norm": float(torch.linalg.vector_norm(embedding)),
        "final_embedding_matches_normalized_scale_concatenation_max_abs_error": pool_match_error,
    })
    final_embeddings.append(F.normalize(embedding, dim=-1))
    scale_vectors_by_series.append(F.normalize(scale_vectors, dim=-1))
    plot_payload.append((plane, patch["energy_cpu"], patch["support_cpu"]))

    del native_volume, canonical, support_canonical, windowed, padded, support_padded
    del model_tensor, gpu_tensor, support_gpu, output
    del captured
    gc.collect()
    torch.cuda.empty_cache()

embedding_matrix = torch.stack(final_embeddings)
final_cosine = embedding_matrix @ embedding_matrix.T
scale_matrix = torch.stack(scale_vectors_by_series)  # [series, scale, feature]
scale_cosines = torch.einsum("isf,jsf->sij", scale_matrix, scale_matrix)

summary = {
    "purpose": "frozen real-volume spatial representation audit; no training",
    "gold_labels_or_gold_studies_used": False,
    "checkpoint_revision": PILLAR_REVISION,
    "input_channels": "one MRI acquisition repeated three times per existing pipeline",
    "preprocessing": "mask-aware linear resampling; intensity window estimated from voxels with support >= 0.5; unsupported voxels zeroed after normalization",
    "series": series_summaries,
    "final_embedding_pairwise_cosine_by_plane_order": final_cosine.tolist(),
    "per_scale_pairwise_cosine_by_plane_order": scale_cosines.tolist(),
    "limitations": [
        "Feature RMS is a descriptive activation statistic, not a causal explanation.",
        "Support fractions mark resampled DICOM source coverage, not a knee anatomy segmentation.",
        "The support map is used for resampling, intensity-window estimation, and audit grouping; it is not supplied as an additional Pillar-0 input channel.",
        "This small report-only sample cannot establish generalization or leaderboard performance.",
    ],
}

(OUTPUT_DIR / "representation_summary.json").write_text(json.dumps(summary, indent=2))
np.savez_compressed(
    OUTPUT_DIR / "pooled_representations.npz",
    final_embeddings=embedding_matrix.numpy(),
    scale_vectors=scale_matrix.numpy(),
    planes=np.asarray([item["plane"] for item in series_summaries]),
)

fig, axes = plt.subplots(len(plot_payload), 2, figsize=(9, 3.2 * len(plot_payload)), squeeze=False)
for index, (plane, energy, support) in enumerate(plot_payload):
    center_d = energy.shape[0] // 2
    axes[index, 0].imshow(energy[center_d].numpy(), cmap="magma")
    axes[index, 0].set_title(f"{plane}: patch-grid feature RMS")
    axes[index, 0].set_xlabel("patch-grid x")
    axes[index, 0].set_ylabel("patch-grid y")
    axes[index, 1].imshow(support[center_d].numpy(), cmap="gray", vmin=0, vmax=1)
    axes[index, 1].set_title(f"{plane}: resampled source-support fraction")
    axes[index, 1].set_xlabel("patch-grid x")
    axes[index, 1].set_ylabel("patch-grid y")
plt.tight_layout()
figure_path = OUTPUT_DIR / "spatial_feature_support.png"
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print(json.dumps({
    "series_count": len(series_summaries),
    "planes": [item["plane"] for item in series_summaries],
    "final_embedding_shapes": [item["final_embedding_shape"] for item in series_summaries],
    "valid_support_fractions": [round(item["valid_dicom_support_fraction"], 4) for item in series_summaries],
    "scale_norm_shares": [item["pooled_scale_norm_shares"] for item in series_summaries],
    "artifact_directory": str(OUTPUT_DIR),
    "saved": ["representation_summary.json", "pooled_representations.npz", figure_path.name],
}, indent=2), flush=True)
